# Echo Epoch Metric Aggregation

Aggregate the per-video SAMWISE Echo epoch scoring outputs into summaries by class, view, and dataset.

This notebook expects the outputs produced by `scripts/eval_echo_epoch_p2flow_metrics.py` and can be pointed at either the `valid` or `external` split.

In [94]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)


In [95]:
EXP_DIR = Path('output/echo_refvos_train')
EPOCH = 1
# SPLIT = 'valid'
SPLIT = 'external'
METHOD_NAME = 'SAMWISE'
PARAMS_M = '-'
FLOPS_G = '-'
DATASET_ORDER = (
    ['CardiacNet', 'HMC_QU', 'Private']
    if SPLIT == 'external'
    else ['Camus', 'CardiacUDA', 'EchoCP', 'SegRWMA', 'EchoNet-Pediatric']
)


def resolve_eval_dir(exp_dir: Path, epoch: int, split: str) -> tuple[Path, Path]:
    candidates: list[tuple[Path, Path]] = []
    if split == 'external':
        external_exp_dir = exp_dir.parent / f'{exp_dir.name}_external_ep{epoch}'
        candidates.append((external_exp_dir, external_exp_dir / 'eval_echo' / split))
    candidates.append((exp_dir, exp_dir / f'valid_epoch{epoch:02d}' / 'eval_echo' / split))
    candidates.append((exp_dir, exp_dir / 'eval_echo' / split))

    for candidate_exp_dir, candidate_eval_dir in candidates:
        if candidate_eval_dir.exists():
            return candidate_exp_dir, candidate_eval_dir

    return candidates[0]


EXP_DIR, VALID_DIR = resolve_eval_dir(EXP_DIR, EPOCH, SPLIT)
EPOCH_DIR = VALID_DIR.parent.parent if VALID_DIR.parent.name == 'eval_echo' else EXP_DIR / f'valid_epoch{EPOCH:02d}'
CSV_PATH = VALID_DIR / 'metrics_p2flow_per_video.csv'
JSON_PATH = VALID_DIR / 'metrics_p2flow_per_video.json'

print('EXP_DIR:', EXP_DIR.resolve())
print('EPOCH_DIR:', EPOCH_DIR.resolve())
print('VALID_DIR:', VALID_DIR.resolve())
print('SPLIT:', SPLIT)
print('CSV_PATH exists:', CSV_PATH.exists())
print('JSON_PATH exists:', JSON_PATH.exists())

EXP_DIR: /home/ultrai/UltrAi/moein/SAMWISE/output/echo_refvos_train_external_ep1
EPOCH_DIR: /home/ultrai/UltrAi/moein/SAMWISE/output/echo_refvos_train_external_ep1
VALID_DIR: /home/ultrai/UltrAi/moein/SAMWISE/output/echo_refvos_train_external_ep1/eval_echo/external
SPLIT: external
CSV_PATH exists: True
JSON_PATH exists: True


In [96]:
def load_metric_rows(csv_path: Path, json_path: Path) -> pd.DataFrame:
    if csv_path.exists():
        df = pd.read_csv(csv_path)
    elif json_path.exists():
        payload = json.loads(json_path.read_text())
        rows = []
        for video_id, video_payload in payload.get('videos', {}).items():
            rows.extend(video_payload.get('records', []))
        df = pd.DataFrame(rows)
    else:
        raise FileNotFoundError(f'Neither CSV nor JSON exists: {csv_path} | {json_path}')

    if df.empty:
        raise ValueError('Loaded metric table is empty.')

    for col in ['video_id', 'dataset', 'view', 'clip_id', 'exp_id', 'obj_id', 'class_name', 'frame']:
        if col in df.columns:
            df[col] = df[col].astype(str)

    for col in ['dice', 'iou', 'hd95']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df


df = load_metric_rows(CSV_PATH, JSON_PATH)
print('Rows:', len(df))
print('Videos:', df['video_id'].nunique())
print('Datasets:', sorted(df['dataset'].unique().tolist()))
print('Views:', sorted(df['view'].unique().tolist()))
print('Classes:', sorted(df['class_name'].unique().tolist()))
display(df.head())

Rows: 33956
Videos: 10567
Datasets: ['CardiacNet', 'EchoNet-Dynamic', 'HMCQU']
Views: ['4CH']
Classes: ['LA', 'LV', 'MYO', 'RA', 'RV']


,video_id,dataset,view,clip_id,exp_id,obj_id,class_name,frame,dice,iou,hd95
0,CardiacNet__4CH__CardiacNet-ASD__ASD__001,CardiacNet,4CH,CardiacNet-ASD__ASD__001,0,1,LV,0,0.929257,0.867863,12.000000
1,CardiacNet__4CH__CardiacNet-ASD__ASD__001,CardiacNet,4CH,CardiacNet-ASD__ASD__001,0,1,LV,1,0.907497,0.830659,13.000000
2,CardiacNet__4CH__CardiacNet-ASD__ASD__001,CardiacNet,4CH,CardiacNet-ASD__ASD__001,0,1,LV,3,0.871704,0.772585,16.000000
3,CardiacNet__4CH__CardiacNet-ASD__ASD__001,CardiacNet,4CH,CardiacNet-ASD__ASD__001,0,1,LV,4,0.860529,0.755200,15.811388
4,CardiacNet__4CH__CardiacNet-ASD__ASD__001,CardiacNet,4CH,CardiacNet-ASD__ASD__001,0,1,LV,5,0.762266,0.615856,27.000000


In [97]:
def summarize_group(frame_df: pd.DataFrame) -> pd.Series:
    out = {'n_rows': int(len(frame_df))}
    for metric in ['dice', 'iou', 'hd95']:
        values = frame_df[metric].replace([np.inf, -np.inf], np.nan).dropna()
        out[f'{metric}_n'] = int(values.shape[0])
        out[f'{metric}_mean'] = float(values.mean()) if len(values) else np.nan
        out[f'{metric}_std'] = float(values.std(ddof=0)) if len(values) else np.nan
    return pd.Series(out)


def summarize_by(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    summary = df.groupby(group_cols, dropna=False, sort=True).apply(summarize_group).reset_index()
    return summary.sort_values(group_cols).reset_index(drop=True)


def add_macro_means(summary_df: pd.DataFrame, metric_prefixes=('dice', 'iou', 'hd95')) -> pd.DataFrame:
    df_out = summary_df.copy()
    for metric in metric_prefixes:
        mean_col = f'{metric}_mean'
        if mean_col in df_out.columns:
            values = pd.to_numeric(df_out[mean_col], errors='coerce')
            df_out.attrs[f'mean_{metric}'] = float(values.dropna().mean()) if values.notna().any() else np.nan
    return df_out


## Per Class

In [98]:
per_class = add_macro_means(summarize_by(df, ['class_name']))
display(per_class)
print({k: v for k, v in per_class.attrs.items() if k.startswith('mean_')})

,class_name,n_rows,dice_n,dice_mean,dice_std,iou_n,iou_mean,iou_std,hd95_n,hd95_mean,hd95_std
0,LA,2341.0,2341.0,0.742460,0.292700,2341.0,0.654291,0.277744,2341.0,35.508854,43.722084
1,LV,24721.0,24721.0,0.844476,0.119926,24721.0,0.743649,0.127213,24633.0,8.525563,14.808186
2,MYO,2283.0,2283.0,0.716664,0.070246,2283.0,0.562956,0.082838,2283.0,9.996971,5.678511
3,RA,2290.0,2290.0,0.817464,0.249935,2290.0,0.742124,0.243737,2290.0,28.449313,34.500060
4,RV,2321.0,2321.0,0.765489,0.245399,2321.0,0.665142,0.229920,2321.0,40.752852,44.234694


{'mean_dice': 0.7773104161097638, 'mean_iou': 0.6736324618886358, 'mean_hd95': 24.646710370985932}


## Per View

In [99]:
per_view = summarize_by(df, ['view'])
display(per_view)

,view,n_rows,dice_n,dice_mean,dice_std,iou_n,iou_mean,iou_std,hd95_n,hd95_mean,hd95_std
0,4CH,33956.0,33956.0,0.821629,0.16373,33956.0,0.719871,0.166683,33868.0,14.045583,25.027775


## Per Dataset

In [100]:
per_dataset = summarize_by(df, ['dataset'])
display(per_dataset)

,dataset,n_rows,dice_n,dice_mean,dice_std,iou_n,iou_mean,iou_std,hd95_n,hd95_mean,hd95_std
0,CardiacNet,9342.0,9342.0,0.767241,0.276366,9342.0,0.680424,0.262116,9260.0,34.333069,41.022835
1,EchoNet-Dynamic,20048.0,20048.0,0.852201,0.068524,20048.0,0.747781,0.090677,20042.0,5.763066,2.962820
2,HMCQU,4566.0,4566.0,0.798670,0.108587,4566.0,0.678035,0.146933,4566.0,9.257163,6.000364


## Per View and Class

In [101]:
per_view_class = summarize_by(df, ['view', 'class_name'])
display(per_view_class)

,view,class_name,n_rows,dice_n,dice_mean,dice_std,iou_n,iou_mean,iou_std,hd95_n,hd95_mean,hd95_std
0,4CH,LA,2341.0,2341.0,0.742460,0.292700,2341.0,0.654291,0.277744,2341.0,35.508854,43.722084
1,4CH,LV,24721.0,24721.0,0.844476,0.119926,24721.0,0.743649,0.127213,24633.0,8.525563,14.808186
2,4CH,MYO,2283.0,2283.0,0.716664,0.070246,2283.0,0.562956,0.082838,2283.0,9.996971,5.678511
3,4CH,RA,2290.0,2290.0,0.817464,0.249935,2290.0,0.742124,0.243737,2290.0,28.449313,34.500060
4,4CH,RV,2321.0,2321.0,0.765489,0.245399,2321.0,0.665142,0.229920,2321.0,40.752852,44.234694


## Per Dataset and Class

In [102]:
per_dataset_class = summarize_by(df, ['dataset', 'class_name'])
display(per_dataset_class)

,dataset,class_name,n_rows,dice_n,dice_mean,dice_std,iou_n,iou_mean,iou_std,hd95_n,hd95_mean,hd95_std
0,CardiacNet,LA,2341.0,2341.0,0.742460,0.292700,2341.0,0.654291,0.277744,2341.0,35.508854,43.722084
1,CardiacNet,LV,2390.0,2390.0,0.745093,0.304600,2390.0,0.661742,0.282560,2308.0,32.522398,39.796719
2,CardiacNet,RA,2290.0,2290.0,0.817464,0.249935,2290.0,0.742124,0.243737,2290.0,28.449313,34.500060
3,CardiacNet,RV,2321.0,2321.0,0.765489,0.245399,2321.0,0.665142,0.229920,2321.0,40.752852,44.234694
4,EchoNet-Dynamic,LV,20048.0,20048.0,0.852201,0.068524,20048.0,0.747781,0.090677,20042.0,5.763066,2.962820
5,HMCQU,LV,2283.0,2283.0,0.880677,0.072094,2283.0,0.793114,0.099147,2283.0,8.517355,6.218409
6,HMCQU,MYO,2283.0,2283.0,0.716664,0.070246,2283.0,0.562956,0.082838,2283.0,9.996971,5.678511


## Paper-Style Table

In [103]:
def build_paper_table(
    per_dataset: pd.DataFrame,
    method_name: str,
    dataset_order: list[str] | None = None,
    split: str = 'valid',
    params_m: str = '-',
    flops_g: str = '-',
) -> tuple[pd.DataFrame, pd.DataFrame]:
    metric_map = {
        'mDice': 'dice_mean',
        'mIoU': 'iou_mean',
        'HD95': 'hd95_mean',
    }

    table_df = per_dataset.copy()
    if split == 'external':
        external_name_map = {
            'HMCQU': 'HMC_QU',
        }
        table_df['dataset'] = table_df['dataset'].replace(external_name_map)

    dataset_to_row = table_df.set_index('dataset').to_dict(orient='index')
    ordered_datasets = []
    if dataset_order:
        ordered_datasets.extend(dataset_order)
    elif split == 'external':
        ordered_datasets.extend(['CardiacNet', 'HMC_QU', 'Private'])
    else:
        ordered_datasets.extend(dataset_to_row.keys())

    numeric_row = {('Method', ''): method_name}
    display_row = {('Method', ''): method_name}
    if split == 'external':
        numeric_row[('Params', '')] = np.nan
        numeric_row[('FLOPs', '')] = np.nan
        display_row[('Params', '')] = params_m
        display_row[('FLOPs', '')] = flops_g
    for dataset in ordered_datasets:
        row = dataset_to_row.get(dataset, {})
        for display_metric, metric_col in metric_map.items():
            value = row.get(metric_col, np.nan)
            if display_metric in {'mDice', 'mIoU'} and pd.notna(value):
                value = value * 100.0
            numeric_row[(dataset, display_metric)] = value
            display_row[(dataset, display_metric)] = '-' if pd.isna(value) else f'{value:.2f}'

    numeric_df = pd.DataFrame([numeric_row])
    display_df = pd.DataFrame([display_row])
    ordered_columns = [('Method', '')]
    if split == 'external':
        ordered_columns.extend([('Params', ''), ('FLOPs', '')])
    for dataset in ordered_datasets:
        ordered_columns.extend([(dataset, 'mDice'), (dataset, 'mIoU'), (dataset, 'HD95')])
    numeric_df = numeric_df.reindex(columns=pd.MultiIndex.from_tuples(ordered_columns))
    display_df = display_df.reindex(columns=pd.MultiIndex.from_tuples(ordered_columns))
    return numeric_df, display_df


paper_table_numeric, paper_table_display = build_paper_table(
    per_dataset,
    METHOD_NAME,
    DATASET_ORDER,
    split=SPLIT,
    params_m=PARAMS_M,
    flops_g=FLOPS_G,
)
display(paper_table_display)

paper_table_latex = paper_table_display.to_latex(index=False, multicolumn=True, multicolumn_format='c', escape=False)
print(paper_table_latex)


Method Params FLOPs CardiacNet               HMC_QU              Private          
                             mDice   mIoU   HD95  mDice   mIoU  HD95   mDice mIoU HD95
0  SAMWISE      -     -      76.72  68.04  34.33  79.87  67.80  9.26       -    -    -

\begin{tabular}{llllllllllll}
\toprule
Method & Params & FLOPs & \multicolumn{3}{c}{CardiacNet} & \multicolumn{3}{c}{HMC_QU} & \multicolumn{3}{c}{Private} \\
 &  &  & mDice & mIoU & HD95 & mDice & mIoU & HD95 & mDice & mIoU & HD95 \\
\midrule
SAMWISE & - & - & 76.72 & 68.04 & 34.33 & 79.87 & 67.80 & 9.26 & - & - & - \\
\bottomrule
\end{tabular}



## Optional Exports

In [104]:
EXPORT_DIR = VALID_DIR / 'aggregated_tables'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

per_class.to_csv(EXPORT_DIR / 'per_class.csv', index=False)
per_view.to_csv(EXPORT_DIR / 'per_view.csv', index=False)
per_dataset.to_csv(EXPORT_DIR / 'per_dataset.csv', index=False)
per_view_class.to_csv(EXPORT_DIR / 'per_view_class.csv', index=False)
per_dataset_class.to_csv(EXPORT_DIR / 'per_dataset_class.csv', index=False)
paper_table_numeric.to_csv(EXPORT_DIR / 'paper_table_numeric.csv', index=False)
paper_table_display.to_csv(EXPORT_DIR / 'paper_table_display.csv', index=False)
(EXPORT_DIR / 'paper_table.tex').write_text(paper_table_latex)

print('Saved aggregated tables to', EXPORT_DIR.resolve())

Saved aggregated tables to /home/ultrai/UltrAi/moein/SAMWISE/output/echo_refvos_train_external_ep1/eval_echo/external/aggregated_tables
